In [7]:
# ============================================================
# CALENDAR ASSISTANT NLU
# COMPLETE INTEGRATED INFERENCE
#
# TASK 1 : Intent Classification
# TASK 2 : Slot Filling
# TASK 3 : Date/Time Extraction
#
# FINAL OUTPUT:
#
# INTENT|BIO TAGS|DATE/TIME
#
# Example:
#
# SET_REMINDER|O O O O O B-PERSON O B-TIME|16:00
# ============================================================


# ============================================================
# 1. MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount(
    "/content/drive"
)


# ============================================================
# 2. IMPORTS
# ============================================================

import os
import json
import random

import torch
import torch.nn as nn


# ============================================================
# 3. DEVICE
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)


# ============================================================
# 4. PATHS
# ============================================================

SAVE_DIR = (
    "/content/drive/MyDrive/"
    "Calendar-Assistant NLU"
)

TASK1_PATH = os.path.join(
    SAVE_DIR,
    "task1_final.pt"
)

TASK2_PATH = os.path.join(
    SAVE_DIR,
    "task2_final.pt"
)

TASK3_PATH = os.path.join(
    SAVE_DIR,
    "task3_final.pt"
)

TRAIN_PATH = os.path.join(
    SAVE_DIR,
    "train.json"
)

TEST_PATH = os.path.join(
    SAVE_DIR,
    "test.json"
)


# ============================================================
# 5. LOAD DATA
# ============================================================

with open(
    TRAIN_PATH,
    "r"
) as f:

    train_data = json.load(f)


with open(
    TEST_PATH,
    "r"
) as f:

    test_data = json.load(f)


print(
    "Number of test examples:",
    len(test_data)
)


# ============================================================
# 6. REBUILD WORD VOCABULARY
# ============================================================

from collections import Counter

counter = Counter()

for sample in train_data:

    counter.update(
        sample["tokens"]
    )


word2idx = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word in counter:

    if word not in word2idx:

        word2idx[word] = len(
            word2idx
        )


idx2word = {
    idx: word
    for word, idx in word2idx.items()
}


print(
    "Vocabulary size:",
    len(word2idx)
)


# ============================================================
# 7. INTENT VOCABULARY
# ============================================================
# ============================================================
# TASK 1 INTENT VOCABULARY
# MUST MATCH TRAINING ORDER
# ============================================================

intent2idx = {
    "CREATE_EVENT": 0,
    "SET_REMINDER": 1,
    "QUERY_FREE_TIME": 2,
    "CANCEL": 3
}

idx2intent = {
    0: "CREATE_EVENT",
    1: "SET_REMINDER",
    2: "QUERY_FREE_TIME",
    3: "CANCEL"
}

print(
    "Intent vocabulary:",
    intent2idx
)



print(
    "Intent vocabulary:",
    intent2idx
)


# ============================================================
# 8. TAG VOCABULARY
# ============================================================

tag2idx = {
    "<pad>": 0
}

for sample in train_data:

    for tag in sample["tags"]:

        if tag not in tag2idx:

            tag2idx[tag] = (
                len(tag2idx)
            )


idx2tag = {
    idx: tag
    for tag, idx in tag2idx.items()
}


print(
    "Tag vocabulary:",
    tag2idx
)


# ============================================================
# ============================================================
# TASK 1
# INTENT CLASSIFICATION
# ============================================================
# ============================================================


# ------------------------------------------------------------
# LOAD CHECKPOINT
# ------------------------------------------------------------

task1_checkpoint = torch.load(
    TASK1_PATH,
    map_location=device
)


if (
    isinstance(task1_checkpoint, dict)
    and
    "model_state_dict"
    in task1_checkpoint
):

    task1_state = (
        task1_checkpoint[
            "model_state_dict"
        ]
    )

else:

    task1_state = task1_checkpoint


# ------------------------------------------------------------
# Infer architecture from checkpoint
# ------------------------------------------------------------

task1_vocab_size = (
    task1_state[
        "embedding.weight"
    ].shape[0]
)

task1_embedding_dim = (
    task1_state[
        "embedding.weight"
    ].shape[1]
)

task1_hidden_dim = (
    task1_state[
        "lstm.weight_ih_l0"
    ].shape[0] // 4
)

task1_num_classes = (
    task1_state[
        "fc.weight"
    ].shape[0]
)


# ------------------------------------------------------------
# MODEL
# ------------------------------------------------------------

class Task1LSTM(
    nn.Module
):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        num_classes,
        pad_idx
    ):

        super().__init__()


        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=pad_idx
        )


        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=False
        )


        self.fc = nn.Linear(
            hidden_dim,
            num_classes
        )


    def forward(
        self,
        X
    ):

        embedded = self.embedding(
            X
        )

        _, (
            hidden,
            cell
        ) = self.lstm(
            embedded
        )

        hidden = hidden[-1]

        return self.fc(
            hidden
        )


# ------------------------------------------------------------
# CREATE MODEL
# ------------------------------------------------------------

model1 = Task1LSTM(
    vocab_size=task1_vocab_size,
    embedding_dim=task1_embedding_dim,
    hidden_dim=task1_hidden_dim,
    num_classes=task1_num_classes,
    pad_idx=word2idx["<PAD>"]
).to(device)


model1.load_state_dict(
    task1_state
)

model1.eval()


print(
    "\nTask 1 loaded:"
)

print(model1)


# ------------------------------------------------------------
# TASK 1 PREDICTION
# ------------------------------------------------------------

def predict_task1(
    tokens
):

    input_ids = [
        word2idx.get(
            word,
            word2idx["<UNK>"]
        )
        for word in tokens
    ]


    X = torch.tensor(
        [input_ids],
        dtype=torch.long
    ).to(device)


    with torch.no_grad():

        output = model1(
            X
        )

        prediction = (
            output
            .argmax(dim=1)
            .item()
        )


    return idx2intent[
        prediction
    ]


# ============================================================
# ============================================================
# TASK 2
# SLOT FILLING
# ============================================================
# ============================================================


# ------------------------------------------------------------
# LOAD CHECKPOINT
# ------------------------------------------------------------

task2_checkpoint = torch.load(
    TASK2_PATH,
    map_location=device
)


if (
    isinstance(task2_checkpoint, dict)
    and
    "model_state_dict"
    in task2_checkpoint
):

    task2_state = (
        task2_checkpoint[
            "model_state_dict"
        ]
    )

else:

    task2_state = task2_checkpoint


# ------------------------------------------------------------
# Infer dimensions
# ------------------------------------------------------------

task2_vocab_size = (
    task2_state[
        "embedding.weight"
    ].shape[0]
)

task2_embedding_dim = (
    task2_state[
        "embedding.weight"
    ].shape[1]
)

task2_hidden_dim = (
    task2_state[
        "lstm.weight_ih_l0"
    ].shape[0] // 4
)

task2_num_tags = (
    task2_state[
        "linear.weight"
    ].shape[0]
)


# ------------------------------------------------------------
# MODEL
# ------------------------------------------------------------

class BiLSTMTagger(
    nn.Module
):

    def __init__(
        self,
        embedding_dim,
        hidden_dim,
        vocab_size,
        num_tags,
        pad_idx
    ):

        super().__init__()


        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=pad_idx
        )


        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )


        self.linear = nn.Linear(
            hidden_dim * 2,
            num_tags
        )


    def forward(
        self,
        X
    ):

        embedded = self.embedding(
            X
        )

        lstm_out, _ = self.lstm(
            embedded
        )

        return self.linear(
            lstm_out
        )


# ------------------------------------------------------------
# CREATE MODEL
# ------------------------------------------------------------

model2 = BiLSTMTagger(
    embedding_dim=task2_embedding_dim,
    hidden_dim=task2_hidden_dim,
    vocab_size=task2_vocab_size,
    num_tags=task2_num_tags,
    pad_idx=word2idx["<PAD>"]
).to(device)


model2.load_state_dict(
    task2_state
)

model2.eval()


print(
    "\nTask 2 loaded:"
)

print(model2)


# ------------------------------------------------------------
# TASK 2 PREDICTION
# ------------------------------------------------------------

def predict_task2(
    tokens
):

    input_ids = [
        word2idx.get(
            word,
            word2idx["<UNK>"]
        )
        for word in tokens
    ]


    X = torch.tensor(
        [input_ids],
        dtype=torch.long
    ).to(device)


    with torch.no_grad():

        output = model2(
            X
        )

        predictions = (
            output
            .argmax(dim=-1)
            [0]
        )


    predicted_tags = [
        idx2tag[
            prediction.item()
        ]
        for prediction in predictions
    ]


    return predicted_tags


# ============================================================
# ============================================================
# TASK 3
# FIVE-COMPONENT ENCODER-DECODER
# ============================================================
# ============================================================


# ------------------------------------------------------------
# LOAD CHECKPOINT
# ------------------------------------------------------------

task3_checkpoint = torch.load(
    TASK3_PATH,
    map_location=device
)


if (
    isinstance(task3_checkpoint, dict)
    and
    "model_state_dict"
    in task3_checkpoint
):

    task3_state = (
        task3_checkpoint[
            "model_state_dict"
        ]
    )

else:

    task3_state = task3_checkpoint


print(
    "\nTask 3 checkpoint keys:"
)

print(
    list(
        task3_state.keys()
    )
)


# ------------------------------------------------------------
# Infer dimensions directly from checkpoint
# ------------------------------------------------------------

task3_vocab_size = (
    task3_state[
        "embedding.weight"
    ].shape[0]
)

task3_embedding_dim = (
    task3_state[
        "embedding.weight"
    ].shape[1]
)

task3_hidden_dim = (
    task3_state[
        "encoder.weight_ih_l0"
    ].shape[0] // 4
)

task3_year_vocab_size = (
    task3_state[
        "year_fc.weight"
    ].shape[0]
)

task3_month_vocab_size = (
    task3_state[
        "month_fc.weight"
    ].shape[0]
)

task3_day_vocab_size = (
    task3_state[
        "day_fc.weight"
    ].shape[0]
)

task3_hour_vocab_size = (
    task3_state[
        "hour_fc.weight"
    ].shape[0]
)

task3_minute_vocab_size = (
    task3_state[
        "minute_fc.weight"
    ].shape[0]
)


# ============================================================
# REBUILD TASK-3 VALUE VOCABULARIES
# ============================================================

# The training code represents missing values as -1.
#
# We therefore construct the exact value ranges used by
# the five output components.
#
# 0 is reserved for "missing".
#
# Actual values are shifted by +1.
#
# Example:
#
# month:
# missing = 0
# January = 1
# February = 2
# ...
#
# hour:
# missing = 0
# 00 = 1
# 01 = 2
# ...
#
# minute:
# missing = 0
# 00 = 1
# 01 = 2
# ...


# ------------------------------------------------------------
# YEAR
# ------------------------------------------------------------

year2idx = {
    None: 0
}

years = set()

for sample in train_data:

    target = (
        sample[
            "target_string"
        ]
        .split("|")[2]
    )


    if target == "NA":
        continue


    parts = target.split(
        " "
    )


    if "-" in parts[0]:

        year = int(
            parts[0]
            .split("-")[0]
        )

        years.add(
            year
        )


for year in sorted(years):

    year2idx[
        year
    ] = len(year2idx)


idx2year = {
    idx: year
    for year, idx in year2idx.items()
}


# ------------------------------------------------------------
# MONTH
# ------------------------------------------------------------

month2idx = {
    None: 0
}

for month in range(
    1,
    13
):

    month2idx[
        month
    ] = month


idx2month = {
    idx: month
    for month, idx in month2idx.items()
}


# ------------------------------------------------------------
# DAY
# ------------------------------------------------------------

day2idx = {
    None: 0
}

for day in range(
    1,
    32
):

    day2idx[
        day
    ] = day


idx2day = {
    idx: day
    for day, idx in day2idx.items()
}


# ------------------------------------------------------------
# HOUR
# ------------------------------------------------------------

hour2idx = {
    None: 0
}

for hour in range(
    0,
    24
):

    hour2idx[
        hour
    ] = hour + 1


idx2hour = {
    idx: hour
    for hour, idx in hour2idx.items()
}


# ------------------------------------------------------------
# MINUTE
# ------------------------------------------------------------

minute2idx = {
    None: 0
}

for minute in range(
    0,
    60
):

    minute2idx[
        minute
    ] = minute + 1


idx2minute = {
    idx: minute
    for minute, idx in minute2idx.items()
}


# ============================================================
# TASK 3 MODEL
# ============================================================

class Task3Seq2Seq(
    nn.Module
):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        year_vocab_size,
        month_vocab_size,
        day_vocab_size,
        hour_vocab_size,
        minute_vocab_size
    ):

        super().__init__()


        # ----------------------------------------------------
        # Encoder
        # ----------------------------------------------------

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )


        self.encoder = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )


        # ----------------------------------------------------
        # Decoder
        # ----------------------------------------------------

        self.decoder = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )


        # ----------------------------------------------------
        # SOS
        #
        # IMPORTANT:
        # checkpoint shape = [1, 300]
        # ----------------------------------------------------

        self.sos_embedding = nn.Parameter(
            torch.randn(
                1,
                embedding_dim
            )
        )


        # ----------------------------------------------------
        # Component embeddings
        # ----------------------------------------------------

        self.year_embedding = nn.Embedding(
            year_vocab_size,
            embedding_dim
        )


        self.month_embedding = nn.Embedding(
            month_vocab_size,
            embedding_dim
        )


        self.day_embedding = nn.Embedding(
            day_vocab_size,
            embedding_dim
        )


        self.hour_embedding = nn.Embedding(
            hour_vocab_size,
            embedding_dim
        )


        self.minute_embedding = nn.Embedding(
            minute_vocab_size,
            embedding_dim
        )


        # ----------------------------------------------------
        # Output heads
        # ----------------------------------------------------

        self.year_fc = nn.Linear(
            hidden_dim,
            year_vocab_size
        )


        self.month_fc = nn.Linear(
            hidden_dim,
            month_vocab_size
        )


        self.day_fc = nn.Linear(
            hidden_dim,
            day_vocab_size
        )


        self.hour_fc = nn.Linear(
            hidden_dim,
            hour_vocab_size
        )


        self.minute_fc = nn.Linear(
            hidden_dim,
            minute_vocab_size
        )


# ============================================================
# CREATE TASK-3 MODEL
# ============================================================

model3 = Task3Seq2Seq(
    vocab_size=task3_vocab_size,
    embedding_dim=task3_embedding_dim,
    hidden_dim=task3_hidden_dim,
    year_vocab_size=task3_year_vocab_size,
    month_vocab_size=task3_month_vocab_size,
    day_vocab_size=task3_day_vocab_size,
    hour_vocab_size=task3_hour_vocab_size,
    minute_vocab_size=task3_minute_vocab_size
).to(device)


# ============================================================
# LOAD TASK-3 WEIGHTS
# ============================================================

model3.load_state_dict(
    task3_state
)

model3.eval()


print(
    "\nTask 3 loaded:"
)

print(model3)


# ============================================================
# PRINT DIMENSIONS
# ============================================================

print(
    "\nTask 3 dimensions:"
)

print(
    "Vocabulary:",
    task3_vocab_size
)

print(
    "Embedding:",
    task3_embedding_dim
)

print(
    "Hidden:",
    task3_hidden_dim
)

print(
    "Year:",
    task3_year_vocab_size
)

print(
    "Month:",
    task3_month_vocab_size
)

print(
    "Day:",
    task3_day_vocab_size
)

print(
    "Hour:",
    task3_hour_vocab_size
)

print(
    "Minute:",
    task3_minute_vocab_size
)


# ============================================================
# TASK 3 PREDICTION
# ============================================================

def predict_task3(
    tokens
):

    # --------------------------------------------------------
    # Convert sentence to word IDs
    # --------------------------------------------------------

    input_ids = [
        word2idx.get(
            word,
            word2idx["<UNK>"]
        )
        for word in tokens
    ]


    X = torch.tensor(
        [input_ids],
        dtype=torch.long
    ).to(device)


    # ========================================================
    # ENCODER
    # ========================================================

    with torch.no_grad():

        embedded = model3.embedding(
            X
        )


        _, (
            hidden,
            cell
        ) = model3.encoder(
            embedded
        )


        # ====================================================
        # STEP 1
        # SOS → YEAR
        # ====================================================

        decoder_input = (
            model3.sos_embedding
            .unsqueeze(0)
        )


        decoder_output, (
            hidden,
            cell
        ) = model3.decoder(
            decoder_input,
            (hidden, cell)
        )


        year_logits = (
            model3.year_fc(
                decoder_output[:, 0, :]
            )
        )


        year_idx = (
            year_logits
            .argmax(dim=-1)
            .item()
        )


        # ====================================================
        # STEP 2
        # YEAR → MONTH
        # ====================================================

        year_token = torch.tensor(
            [year_idx],
            dtype=torch.long,
            device=device
        )


        decoder_input = (
            model3.year_embedding(
                year_token
            )
            .unsqueeze(1)
        )


        decoder_output, (
            hidden,
            cell
        ) = model3.decoder(
            decoder_input,
            (hidden, cell)
        )


        month_logits = (
            model3.month_fc(
                decoder_output[:, 0, :]
            )
        )


        month_idx = (
            month_logits
            .argmax(dim=-1)
            .item()
        )


        # ====================================================
        # STEP 3
        # MONTH → DAY
        # ====================================================

        month_token = torch.tensor(
            [month_idx],
            dtype=torch.long,
            device=device
        )


        decoder_input = (
            model3.month_embedding(
                month_token
            )
            .unsqueeze(1)
        )


        decoder_output, (
            hidden,
            cell
        ) = model3.decoder(
            decoder_input,
            (hidden, cell)
        )


        day_logits = (
            model3.day_fc(
                decoder_output[:, 0, :]
            )
        )


        day_idx = (
            day_logits
            .argmax(dim=-1)
            .item()
        )


        # ====================================================
        # STEP 4
        # DAY → HOUR
        # ====================================================

        day_token = torch.tensor(
            [day_idx],
            dtype=torch.long,
            device=device
        )


        decoder_input = (
            model3.day_embedding(
                day_token
            )
            .unsqueeze(1)
        )


        decoder_output, (
            hidden,
            cell
        ) = model3.decoder(
            decoder_input,
            (hidden, cell)
        )


        hour_logits = (
            model3.hour_fc(
                decoder_output[:, 0, :]
            )
        )


        hour_idx = (
            hour_logits
            .argmax(dim=-1)
            .item()
        )


        # ====================================================
        # STEP 5
        # HOUR → MINUTE
        # ====================================================

        hour_token = torch.tensor(
            [hour_idx],
            dtype=torch.long,
            device=device
        )


        decoder_input = (
            model3.hour_embedding(
                hour_token
            )
            .unsqueeze(1)
        )


        decoder_output, (
            hidden,
            cell
        ) = model3.decoder(
            decoder_input,
            (hidden, cell)
        )


        minute_logits = (
            model3.minute_fc(
                decoder_output[:, 0, :]
            )
        )


        minute_idx = (
            minute_logits
            .argmax(dim=-1)
            .item()
        )


    # ========================================================
    # CONVERT BACK TO ACTUAL VALUES
    # ========================================================

    year = idx2year.get(
        year_idx,
        None
    )

    month = idx2month.get(
        month_idx,
        None
    )

    day = idx2day.get(
        day_idx,
        None
    )

    hour = idx2hour.get(
        hour_idx,
        None
    )

    minute = idx2minute.get(
        minute_idx,
        None
    )


    # ========================================================
    # DETERMINE OUTPUT TYPE
    # ========================================================

    has_date = (
        year is not None
        and
        month is not None
        and
        day is not None
    )


    has_time = (
        hour is not None
        and
        minute is not None
    )


    # --------------------------------------------------------
    # DATE + TIME
    # --------------------------------------------------------

    if has_date and has_time:

        return (
            f"{year:04d}-"
            f"{month:02d}-"
            f"{day:02d} "
            f"{hour:02d}:"
            f"{minute:02d}"
        )


    # --------------------------------------------------------
    # DATE ONLY
    # --------------------------------------------------------

    if has_date:

        return (
            f"{year:04d}-"
            f"{month:02d}-"
            f"{day:02d}"
        )


    # --------------------------------------------------------
    # TIME ONLY
    # --------------------------------------------------------

    if has_time:

        return (
            f"{hour:02d}:"
            f"{minute:02d}"
        )


    # --------------------------------------------------------
    # NOTHING
    # --------------------------------------------------------

    return "NA"


# ============================================================
# COMPLETE PIPELINE
# ============================================================

def run_calendar_assistant(
    sample
):

    tokens = sample["tokens"]


    # --------------------------------------------------------
    # TASK 1
    # --------------------------------------------------------

    intent = predict_task1(
        tokens
    )


    # --------------------------------------------------------
    # TASK 2
    # --------------------------------------------------------

    tags = predict_task2(
        tokens
    )


    # --------------------------------------------------------
    # TASK 3
    # --------------------------------------------------------

    datetime_output = predict_task3(
        tokens
    )


    # --------------------------------------------------------
    # FINAL FORMAT
    # --------------------------------------------------------

    return (
        intent
        + "|"
        + " ".join(tags)
        + "|"
        + datetime_output
    )


# ============================================================
# TEST 10 RANDOM EXAMPLES
# ============================================================

print("\n")
print("=" * 80)
print(
    "INTEGRATED CALENDAR ASSISTANT NLU"
)
print("=" * 80)


random_samples = random.sample(
    test_data,
    10
)


for i, sample in enumerate(
    random_samples,
    start=1
):

    print("\n")
    print(
        f"EXAMPLE {i}"
    )

    print(
        "-" * 80
    )


    print(
        "Input:"
    )

    print(
        sample["raw_text"]
    )


    actual = (
        sample[
            "target_string"
        ]
    )


    predicted = (
        run_calendar_assistant(
            sample
        )
    )


    print(
        "\nActual:"
    )

    print(
        actual
    )


    print(
        "\nPredicted:"
    )

    print(
        predicted
    )


    print(
        "-" * 80
    )


print("\n")
print("=" * 80)
print(
    "INTEGRATED TEST COMPLETE"
)
print("=" * 80)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cpu
Number of test examples: 725
Vocabulary size: 649
Intent vocabulary: {'CREATE_EVENT': 0, 'SET_REMINDER': 1, 'QUERY_FREE_TIME': 2, 'CANCEL': 3}
Intent vocabulary: {'CREATE_EVENT': 0, 'SET_REMINDER': 1, 'QUERY_FREE_TIME': 2, 'CANCEL': 3}
Tag vocabulary: {'<pad>': 0, 'O': 1, 'B-PERSON': 2, 'B-TIME': 3, 'B-EVENT': 4, 'I-EVENT': 5, 'B-DATE': 6, 'I-DATE': 7}

Task 1 loaded:
Task1LSTM(
  (embedding): Embedding(649, 300, padding_idx=0)
  (lstm): LSTM(300, 256, batch_first=True)
  (fc): Linear(in_features=256, out_features=4, bias=True)
)

Task 2 loaded:
BiLSTMTagger(
  (embedding): Embedding(649, 300, padding_idx=0)
  (lstm): LSTM(300, 128, batch_first=True, bidirectional=True)
  (linear): Linear(in_features=256, out_features=8, bias=True)
)

Task 3 checkpoint keys:
['sos_embedding', 'embedding.weight', 'encoder.weight_ih_l0', 'encoder.weight_hh_l0', 'enc